In [ ]:
# Mount Drive (Colab)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install -U tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 113.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.19.0 requires tensorflow<2.20,>=2.19, but you have tensorflow 2.20.0 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.20.0 which is incompatible.
tensorflow-text 2.19.0 requires tensorflow<2.20,>=2.19.0, but you have tensorflow 2.20.0 which is incompatible.


In [ ]:
import os, time, json, pathlib, itertools
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

# ============================================================
# CONFIG (EDIT ONLY HERE)
# ============================================================
DATASET_DIR = "/content/drive/MyDrive/ImageClassification/TrashData"
OUT_DIR     = "/content/drive/MyDrive/ImageClassification/OUT_InceptionV3"

# InceptionV3 standard input = 299x299
IMG_SIZE    = (299, 299)

LR_HEAD = 1e-3
LR_FINE = 1e-5

BATCH_PER_REPLICA = 32 #32
SEED             = 42
KFOLDS           = 2
EPOCHS_HEAD      = 5
EPOCHS_FINE      = 8  #10
CV_PATIENCE      = 4
FINAL_PATIENCE   = 5

WARMUP_RUNS      = 5  #7
BENCH_RUNS       = 8  #10
REP_BATCHES      = 10 #15

TEST_SIZE        = 0.30
FINAL_VAL_SIZE   = 0.15
FINE_TUNE_RATIO  = 0.8

FOCAL_GAMMA      = 2.0
FOCAL_ALPHA      = 1.0
REDUCE_LR_FACTOR = 0.5

AUTOTUNE = tf.data.AUTOTUNE

# ============================================================
# OUTPUT PATHS
# ============================================================
os.makedirs(OUT_DIR, exist_ok=True)
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

MODEL_PATH       = os.path.join(OUT_DIR, "best.keras")
CLEAN_MODEL_PATH = os.path.join(OUT_DIR, "best_noopt.keras")
METRICS_PATH     = os.path.join(OUT_DIR, "metrics.json")

LOSS_PNG = os.path.join(PLOTS_DIR, "loss_train_vs_val.png")
ACC_PNG  = os.path.join(PLOTS_DIR, "acc_train_vs_val.png")
CM_PNG   = os.path.join(PLOTS_DIR, "confusion_matrix.png")

TFLITE_FP32_PATH = os.path.join(OUT_DIR, "best_fp32.tflite")
TFLITE_DRQ_PATH  = os.path.join(OUT_DIR, "best_dynamic_range.tflite")
TFLITE_INT8_PATH = os.path.join(OUT_DIR, "best_int8.tflite")
LABELS_TXT       = os.path.join(OUT_DIR, "labels.txt")

np.random.seed(SEED)
tf.random.set_seed(SEED)
# !pip install tensorflow

In [ ]:
# ============================================================
# TPU / STRATEGY INIT
# ============================================================
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print("TPU:", tpu.master())
except Exception as e:
    strategy = tf.distribute.get_strategy()
    print("TPU not used. Fallback strategy. Reason:", e)

print("Strategy:", type(strategy).__name__)
print("Replicas:", strategy.num_replicas_in_sync)

NUM_REPLICAS = strategy.num_replicas_in_sync
GLOBAL_BATCH = BATCH_PER_REPLICA * NUM_REPLICAS
print("GLOBAL_BATCH:", GLOBAL_BATCH)

TPU not used. Fallback strategy. Reason: Please provide a TPU Name to connect to.
Strategy: _DefaultDistributionStrategy
Replicas: 1
GLOBAL_BATCH: 32


In [ ]:
# ============================================================
# DATA HELPERS
# ============================================================
def list_files(dataset_dir):
    p = pathlib.Path(dataset_dir)
    classes = sorted([d.name for d in p.iterdir() if d.is_dir()])
    paths, labels = [], []
    for i, c in enumerate(classes):
        for x in (p / c).glob("*"):
            if x.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                paths.append(str(x))
                labels.append(i)
    return np.array(paths), np.array(labels), classes

def decode_resize(path, label):
    img_bytes = tf.io.read_file(path)
    is_jpeg = tf.strings.regex_full_match(tf.strings.lower(path), ".*\\.jpe?g")
    img = tf.cond(
        is_jpeg,
        lambda: tf.image.decode_jpeg(img_bytes, channels=3),
        lambda: tf.image.decode_png(img_bytes, channels=3),
    )
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    img = preprocess_input(img)   # InceptionV3 preprocess
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.08)
    img = tf.image.random_contrast(img, 0.9, 1.1)
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    img = tf.image.rot90(img, k)
    return img, label

def make_ds(paths, labels, train=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if train:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_resize, num_parallel_calls=AUTOTUNE)
    if train:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(GLOBAL_BATCH, drop_remainder=train).prefetch(AUTOTUNE)
    return ds

def oversample(paths, labels):
    rng = np.random.RandomState(SEED)
    cls, cnt = np.unique(labels, return_counts=True)
    m = cnt.max()
    new_p, new_y = [], []
    for c in cls:
        idx = np.where(labels == c)[0]
        chosen = rng.choice(idx, size=m, replace=True)
        new_p.extend(paths[chosen].tolist())
        new_y.extend([c] * m)
    perm = rng.permutation(len(new_p))
    return np.array(new_p)[perm], np.array(new_y)[perm]

In [ ]:
# ============================================================
# MODEL
# ============================================================
def focal_loss(num_classes, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_one = tf.one_hot(y_true, depth=num_classes)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        ce = -y_one * tf.math.log(y_pred)
        w = alpha * tf.math.pow(1 - y_pred, gamma)
        return tf.reduce_sum(w * ce, axis=1)
    return loss

def build_model(num_classes):
    base = InceptionV3(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights="imagenet"
    )
    base.trainable = False

    inp = layers.Input(shape=IMG_SIZE + (3,))
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(inp, out), base

def compile_model(model, num_classes, lr):
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss=focal_loss(num_classes),
        metrics=["accuracy"],
    )

def fine_tune(base, ratio=FINE_TUNE_RATIO):
    base.trainable = True
    n = len(base.layers)
    fine_at = int(n * ratio)
    for i, layer in enumerate(base.layers):
        layer.trainable = (i >= fine_at)

In [ ]:
# ============================================================
# EVAL
# ============================================================
def eval_metrics(model, ds, class_names):
    y_true, y_pred = [], []
    for x, y in ds:
        p = model.predict(x, verbose=0)
        y_true.extend(y.numpy().tolist())
        y_pred.extend(np.argmax(p, axis=1).tolist())

    rep = classification_report(
        y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0
    )
    acc = rep["accuracy"]
    prec = np.mean([rep[c]["precision"] for c in class_names])
    rec  = np.mean([rep[c]["recall"] for c in class_names])
    f1   = np.mean([rep[c]["f1-score"] for c in class_names])
    return acc, prec, rec, f1

def get_preds(model, ds):
    y_true, y_pred = [], []
    for x, y in ds:
        p = model.predict(x, verbose=0)
        y_true.extend(y.numpy().tolist())
        y_pred.extend(np.argmax(p, axis=1).tolist())
    return np.array(y_true), np.array(y_pred)

# ============================================================
# SAVE 2 GRAPHS
# ============================================================
def save_train_val_plots(h1, h2, loss_path, acc_path):
    def cat(k):
        return list(h1.history.get(k, [])) + list(h2.history.get(k, []))

    tr_loss = cat("loss")
    va_loss = cat("val_loss")
    tr_acc  = cat("accuracy")
    va_acc  = cat("val_accuracy")
    epochs  = np.arange(1, len(tr_loss) + 1)

    plt.figure()
    plt.plot(epochs, tr_loss, label="train_loss")
    plt.plot(epochs, va_loss, label="val_loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend(); plt.grid(True)
    plt.tight_layout()
    plt.savefig(loss_path, dpi=200)
    plt.close()

    plt.figure()
    plt.plot(epochs, tr_acc, label="train_acc")
    plt.plot(epochs, va_acc, label="val_acc")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy")
    plt.title("Training vs Validation Accuracy")
    plt.legend(); plt.grid(True)
    plt.tight_layout()
    plt.savefig(acc_path, dpi=200)
    plt.close()

# ============================================================
# CONFUSION MATRIX SAVE
# (blue cmap, text always black)
# ============================================================
def save_confusion_matrix_png(y_true, y_pred, class_names, out_path, normalize=False):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))

    if normalize:
        cm = cm.astype("float32") / np.maximum(cm.sum(axis=1, keepdims=True), 1e-9)

    plt.figure(figsize=(8, 8))

    vmax = 1.0 if normalize else cm.max()
    plt.imshow(cm, cmap="Blues", vmin=0.0, vmax=vmax)

    plt.title("Confusion Matrix" + (" (Normalized)" if normalize else ""))
    plt.colorbar()

    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45, ha="right")
    plt.yticks(tick_marks, class_names)

    fmt = ".2f" if normalize else "d"

    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(
            j, i, format(cm[i, j], fmt),
            ha="center",
            va="center",
            color="black"
        )

    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

In [8]:
# ============================================================
# RUN
# ============================================================
paths, labels, class_names = list_files(DATASET_DIR)
num_classes = len(class_names)
print("Classes:", class_names)
print("Num classes:", num_classes)

# Split: Train vs Test (TEST never touched in CV/training decisions)
train_p, test_p, train_y, test_y = train_test_split(
    paths, labels, test_size=TEST_SIZE, random_state=SEED, stratify=labels
)

# -----------------------
# CV on TRAIN only
# -----------------------
skf = StratifiedKFold(n_splits=KFOLDS, shuffle=True, random_state=SEED)
cv_acc = []

for fold, (tr, va) in enumerate(skf.split(train_p, train_y), start=1):
    tr_p, tr_l = train_p[tr], train_y[tr]
    va_p, va_l = train_p[va], train_y[va]

    tr_p2, tr_l2 = oversample(tr_p, tr_l)
    tr_ds = make_ds(tr_p2, tr_l2, train=True)
    va_ds = make_ds(va_p, va_l, train=False)

    with strategy.scope():
        model, base = build_model(num_classes)
        compile_model(model, num_classes, LR_HEAD)

    cb_cv = [
        callbacks.EarlyStopping(monitor="val_loss", patience=CV_PATIENCE, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=REDUCE_LR_FACTOR, patience=1),
    ]

    model.fit(tr_ds, validation_data=va_ds, epochs=EPOCHS_HEAD, verbose=1, callbacks=cb_cv)

    with strategy.scope():
        fine_tune(base, ratio=FINE_TUNE_RATIO)
        compile_model(model, num_classes, LR_FINE)

    model.fit(tr_ds, validation_data=va_ds, epochs=EPOCHS_FINE, verbose=1, callbacks=cb_cv)

    acc, _, _, _ = eval_metrics(model, va_ds, class_names)
    cv_acc.append(acc)
    print(f"Fold {fold} val accuracy: {acc:.4f}")

cv_mean, cv_std = float(np.mean(cv_acc)), float(np.std(cv_acc))
print(f"\nCV Accuracy (mean ± std): {cv_mean:.4f} ± {cv_std:.4f}")

# -----------------------
# Final Train/Val split from TRAIN only
# -----------------------
final_tr_p, final_va_p, final_tr_y, final_va_y = train_test_split(
    train_p, train_y, test_size=FINAL_VAL_SIZE, random_state=SEED, stratify=train_y
)

final_tr_p2, final_tr_y2 = oversample(final_tr_p, final_tr_y)
final_tr_ds = make_ds(final_tr_p2, final_tr_y2, train=True)
final_va_ds = make_ds(final_va_p, final_va_y, train=False)
test_ds     = make_ds(test_p, test_y, train=False)

# -----------------------
# Final training + checkpoint
# -----------------------
with strategy.scope():
    model, base = build_model(num_classes)
    compile_model(model, num_classes, LR_HEAD)

ckpt = callbacks.ModelCheckpoint(MODEL_PATH, monitor="val_accuracy", save_best_only=True, verbose=1)
cb_final = [
    ckpt,
    callbacks.EarlyStopping(monitor="val_loss", patience=FINAL_PATIENCE, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=REDUCE_LR_FACTOR, patience=1),
]

h1 = model.fit(final_tr_ds, validation_data=final_va_ds, epochs=EPOCHS_HEAD, verbose=1, callbacks=cb_final)

with strategy.scope():
    fine_tune(base, ratio=FINE_TUNE_RATIO)
    compile_model(model, num_classes, LR_FINE)

h2 = model.fit(final_tr_ds, validation_data=final_va_ds, epochs=EPOCHS_FINE, verbose=1, callbacks=cb_final)

# Load best + save clean model (no optimizer)
best = tf.keras.models.load_model(MODEL_PATH, compile=False)
best.save(CLEAN_MODEL_PATH, include_optimizer=False)

# Save plots
save_train_val_plots(h1, h2, LOSS_PNG, ACC_PNG)

# Test evaluation
test_acc, test_prec, test_rec, test_f1 = eval_metrics(best, test_ds, class_names)

# Confusion matrix (normalized)
y_true, y_pred = get_preds(best, test_ds)
save_confusion_matrix_png(y_true, y_pred, class_names, CM_PNG, normalize=True)

# Model size
train_model_size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
clean_model_size_mb = os.path.getsize(CLEAN_MODEL_PATH) / (1024 * 1024)

Classes: ['Biological', 'Metal', 'Paper', 'Plastic']
Num classes: 4
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
43/43 ━━━━━━━━━━━━━━━━━━━━ 518s 12s/step - accuracy: 0.5081 - loss: 0.6790 - val_accuracy: 0.8932 - val_loss: 0.1576 - learning_rate: 0.0010
Epoch 2/5
43/43 ━━━━━━━━━━━━━━━━━━━━ 424s 10s/step - accuracy: 0.8475 - loss: 0.1814 - val_accuracy: 0.9104 - val_loss: 0.1171 - learning_rate: 0.0010
Epoch 3/5
43/43 ━━━━━━━━━━━━━━━━━━━━ 495s 11s/step - accuracy: 0.8955 - loss: 0.1256 - val_accuracy: 0.9211 - val_loss: 0.1028 - learning_rate: 0.0010
Epoch 4/5
43/43 ━━━━━━━━━━━━━━━━━━━━ 477s 11s/step - accuracy: 0.8973 - loss: 0.1107 - val_accuracy: 0.9233 - val_loss: 0.0961 - learning_rate: 0.0010
Epoch 5/5
43/43 ━━━━━━━━━━━━━━━━━━━━ 473s 11s/step - accuracy: 0.9220 - loss: 0.0833 - val_accuracy: 0.9262 - val_loss: 0.0953 - learning_rate: 0.0010
Epoch 1/8
43/43 ━━━━━━━━━━━━━━━━━━━━ 555s 13s/step - accuracy: 0.8851 - loss: 0.1426 - val_accuracy: 0.9319 - val_loss: 0.0898

In [9]:
# ============================================================
# BENCHMARK
# ============================================================
def benchmark_warm(model, warmup=WARMUP_RUNS, runs=BENCH_RUNS):
    x1 = np.random.rand(1, *IMG_SIZE, 3).astype("float32")
    xb = np.random.rand(GLOBAL_BATCH, *IMG_SIZE, 3).astype("float32")

    for _ in range(warmup):
        _ = model.predict(x1, verbose=0)

    t0 = time.time()
    for _ in range(runs):
        _ = model.predict(x1, verbose=0)
    single_ms = ((time.time() - t0) / runs) * 1000

    for _ in range(warmup):
        _ = model.predict(xb, verbose=0)

    t0 = time.time()
    for _ in range(runs):
        _ = model.predict(xb, verbose=0)
    total = time.time() - t0

    thr = (GLOBAL_BATCH * runs) / total
    eff_ms = 1000.0 / thr
    return single_ms, thr, eff_ms

single_ms, thr, eff_ms = benchmark_warm(best, warmup=WARMUP_RUNS, runs=BENCH_RUNS)
params = int(best.count_params())

out = {
    "model": "InceptionV3",
    "classes": class_names,
    "strategy": type(strategy).__name__,
    "replicas": int(NUM_REPLICAS),
    "global_batch": int(GLOBAL_BATCH),
    "cv_accuracy_mean": cv_mean,
    "cv_accuracy_std": cv_std,
    "test_accuracy": float(test_acc),
    "precision_macro": float(test_prec),
    "recall_macro": float(test_rec),
    "f1_macro": float(test_f1),
    "parameters": params,
    "model_size_mb_training_checkpoint": float(train_model_size_mb),
    "model_size_mb_clean_inference": float(clean_model_size_mb),
    "single_image_latency_ms_after_warmup": float(single_ms),
    "throughput_images_per_sec": float(thr),
    "effective_ms_per_image_from_throughput": float(eff_ms),
    "loss_plot_path": LOSS_PNG,
    "accuracy_plot_path": ACC_PNG,
    "confusion_matrix_plot_path": CM_PNG,
}

with open(METRICS_PATH, "w") as f:
    json.dump(out, f, indent=2)

print("\nSaved training checkpoint:", MODEL_PATH)
print("Saved clean inference model:", CLEAN_MODEL_PATH)
print("Saved loss plot:", LOSS_PNG)
print("Saved accuracy plot:", ACC_PNG)
print("Saved confusion matrix:", CM_PNG)
print("Metrics:", METRICS_PATH)


Saved training checkpoint: /content/drive/MyDrive/ImageClassification/OUT_InceptionV3/best.keras
Saved clean inference model: /content/drive/MyDrive/ImageClassification/OUT_InceptionV3/best_noopt.keras
Saved loss plot: /content/drive/MyDrive/ImageClassification/OUT_InceptionV3/plots/loss_train_vs_val.png
Saved accuracy plot: /content/drive/MyDrive/ImageClassification/OUT_InceptionV3/plots/acc_train_vs_val.png
Saved confusion matrix: /content/drive/MyDrive/ImageClassification/OUT_InceptionV3/plots/confusion_matrix.png
Metrics: /content/drive/MyDrive/ImageClassification/OUT_InceptionV3/metrics.json


In [10]:
# ============================================================
# TFLite EXPORT
# ============================================================
with open(LABELS_TXT, "w") as f:
    for name in class_names:
        f.write(name + "\n")
print("Saved labels:", LABELS_TXT)

clean = tf.keras.models.load_model(CLEAN_MODEL_PATH, compile=False)

# FP32
converter = tf.lite.TFLiteConverter.from_keras_model(clean)
tflite_fp32 = converter.convert()
with open(TFLITE_FP32_PATH, "wb") as f:
    f.write(tflite_fp32)
print("Saved TFLite FP32:", TFLITE_FP32_PATH)

# Dynamic Range Quant
converter = tf.lite.TFLiteConverter.from_keras_model(clean)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_drq = converter.convert()
with open(TFLITE_DRQ_PATH, "wb") as f:
    f.write(tflite_drq)
print("Saved TFLite Dynamic Range:", TFLITE_DRQ_PATH)

# INT8
def rep_data_gen():
    for x, _ in final_tr_ds.take(REP_BATCHES):
        yield [tf.cast(x, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(clean)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = rep_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

try:
    tflite_int8 = converter.convert()
    with open(TFLITE_INT8_PATH, "wb") as f:
        f.write(tflite_int8)
    print("Saved TFLite INT8:", TFLITE_INT8_PATH)
except Exception as e:
    print("INT8 conversion failed. Reason:", e)
    print("Tip: remove int8 input/output lines for hybrid quantization and try again.")

Saved labels: /content/drive/MyDrive/ImageClassification/OUT_InceptionV3/labels.txt
Saved artifact at '/tmp/tmpupsx56kf'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 299, 299, 3), dtype=tf.float32, name='input_layer_5')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  135121927271248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135121927267216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135121927270480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135121927274128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135121927268368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135121927269904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135121927272400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135121927272208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135121927273552: TensorSpec(shape=(), dtype=tf.resource, name=None)
 

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Saved TFLite INT8: /content/drive/MyDrive/ImageClassification/OUT_InceptionV3/best_int8.tflite
